In [1]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import numpy as np
import math
import pygame
import matplotlib.pyplot as plt
import threading
import time


"""
Define colors and constants.
"""

# constants
G = 6.67430e-11
mass_sun = 1.99e30
mass_mars = 6.39e26
mass_jupiter = 10.89e27
mass_earth = 5.97e24

radius_earth = 6e6
radius_mars = 5e6
radius_sun = 7e8
radius_jupiter = 8e7

luminosity_sun = 3.9e26

# colors
red = (255, 0, 0)
orange = (255, 128, 0)
yellow = (255, 255, 0)
black = (0,0,0)
white = (255,255,255)
blue = (0,0,255)
cadmiumorange = (255,97,3)
azure4 = (131,139,139)
green = (0,255,0)
grey = (128,128,128)

hotpink1 = (255, 110, 180)  
hotpink2 = (238, 106, 167)  
hotpink3 = (205, 96, 144)   

purple = (128,0,128)

# for more colors:
# https://www.webucator.com/article/python-color-constants-module/ 

# In[2]:

"""
This is the Body class, which includes the information for each Body as well as all of the physics
created by the program (and a function to detect where the mouse is in correlation to the bodies.
"""
class Body:
    def __init__(self, name, mass, position, velocity, color, radius, luminosity):
        self.name = name
        self.mass = mass
        self.position = np.array(position, dtype='float64')
        self.velocity = np.array(velocity, dtype='float64')
        self.acceleration = np.zeros(2)
        self.color = color
        self.radius = radius
        self.luminosity = luminosity

    def update(self, dt):
        self.velocity += self.acceleration * dt
        self.position += self.velocity * dt

    def calculate_acceleration(self, bodies):
        total_force = np.zeros(2)
        for body in bodies:
            if body is not self:
                r_vector = body.position - self.position
                r = np.linalg.norm(r_vector)
                force_magnitude = G * self.mass * body.mass / r**2
                force_direction = r_vector / r
                total_force += force_magnitude * force_direction
        self.acceleration = total_force / self.mass

    def is_mouse_over(self, mouse_pos, scale, sun_center):
        body_pos_screen = sun_center + (self.position / scale)
        distance = np.linalg.norm(mouse_pos - body_pos_screen)
        return distance < 10 

    def is_star(self):
        return self.luminosity > 0

    def calculate_intensity(self, observer_position, other_bodies, scale):
        if not self.is_star():
            return 0  
        
        d = np.linalg.norm(self.position - observer_position)
        intensity = self.luminosity / (4 * math.pi * d**2)

        for body in other_bodies:
            if body is not self and body is not observer_position:
                if self.is_blocking(body, observer_position, scale):
                    obscured_area = np.pi * (body.radius / scale)**2
                    star_area = np.pi * (self.radius / scale)**2
                    intensity *= (1 - obscured_area/star_area)
        return intensity

    def is_blocking(self, other, observer_position, scale):
        other_pos = other.position - observer_position
        self_pos = self.position - observer_position
        distance_to_line = np.linalg.norm(np.cross(self_pos, other_pos)) / np.linalg.norm(self_pos)
        return distance_to_line < other.radius / scale
    """
    def intensity(d):
        d = np.linalg.norm(self.position
        return self.luminosity / 4 * math.pi * d**2
        """

# In[3]:


# design celestial bodies

""" Solar System - Real One

sun = Body("Sun", mass_sun, [0, 0], [0, 0], yellow, radius_sun, luminosity_sun)
mars = Body("Mars", mass_mars, [227e9, 300], [0, 24e3], red, radius_mars,0)  
jupiter = Body("Jupiter", mass_jupiter, [100e9, 0], [0, 13e3], orange, radius_jupiter,0) 
earth = Body("Earth", mass_earth, [-149e9,0],[0,-30e3],blue, radius_earth,0)
#sun2 = Body("Second Sun", mass_sun, [100e3, 100e3], [0,0], white)
alpha_centauri = Body("Alpha Centauri", 2*mass_sun, [100e10,0],[0,0],cadmiumorange, 2*radius_sun, 2*luminosity_sun)
dark_matter = Body("Dark Matter", 100000*mass_sun, [100e40,0],[0,0], azure4, 10*radius_sun, 0)

"""

yggrdasil = Body("Yggdrasil", mass_sun, [0, 0], [0, 0], yellow, radius_sun, luminosity_sun)
ymir = Body("Ymir", 0.01*mass_sun, [300e9, 0], [0, -20e3], cadmiumorange, 0.5*radius_sun, 0)
ve = Body("Ve", 0.5*mass_earth, [107e9,0],[0,-30e3],grey, 0.5*radius_earth,0)
tyr = Body("Tyr", 0.75*mass_earth, [0,-130e9],[-30e3,0],orange, 0.75*radius_earth,0)
freyja = Body("Freyja", 1.1*mass_earth, [-149e9,0],[0,30e3],green,1.1*radius_earth,0)
forsetti = Body("Forsetti", 3*mass_jupiter, [-500e9, 0], [0, 20e3], azure4, 3*radius_jupiter,0) 

# comets
kvasir = Body("Kvasir", 0.000001*mass_earth, [30e11,30e9], [-30e10,-30e10], purple,0.01*radius_earth,0)

# declare celestial bodies
bodies = [yggrdasil,ymir,ve,tyr,freyja,forsetti,kvasir]


# In[4]:


# pygame setup
pygame.init()
width, height = 800, 600
screen = pygame.display.set_mode((width, height))
clock = pygame.time.Clock()
running = True
paused = False
scale = 1e9  
sun_center = np.array([width // 2, height // 2])

def draw_body(body):
    position = sun_center + (body.position / scale)
    pygame.draw.circle(screen, body.color, position.astype(int), 5)


def real_time_plotting(intensity_values):
    plt.ion()
    figures = {star_name: plt.subplots() for star_name in star_names}
    lines = {star_name: ax.plot([], [])[0] for star_name, (fig, ax) in figures.items()}

    while True:
        for star_name in star_names:
            if len(intensity_values[star_name]) > 0:
                lines[star_name].set_xdata(list(range(len(intensity_values[star_name]))))
                lines[star_name].set_ydata(intensity_values[star_name])
                figures[star_name][1].relim()
                figures[star_name][1].autoscale_view()
                figures[star_name][0].canvas.draw()
                figures[star_name][0].canvas.flush_events()
        time.sleep(0.1)  # Update every 0.1 seconds


# In[5]:

# draw info box and speed box

def draw_info_box(body):
    font = pygame.font.Font(None, 24)
    info_text = [
        f"Name: {body.name}",
        f"X: {body.position[0]:.2e}",
        f"Y: {body.position[1]:.2e}",
        f"v: {np.linalg.norm(body.velocity):.2e}",
        f"a: {np.linalg.norm(body.acceleration):.2e}"
    ]
    box_width = max(font.size(line)[0] for line in info_text) + 20
    box_height = len(info_text) * font.get_linesize()

    # Draw a rectangle as the background of the info box
    box_rect = pygame.Rect(10, 10, box_width, box_height)
    screen.fill((0, 0, 0), box_rect)
    pygame.draw.rect(screen, (0, 0, 0), box_rect, 1)

    for i, line in enumerate(info_text):
        text_surface = font.render(line, True, (255, 255, 255))
        screen.blit(text_surface, (15, 15 + i * font.get_linesize()))

#### dt = 10000  # Time step in seconds

speed_settings = [5000, 10000, 15000, 20000, 250000]  # Different time steps for simulation speed
current_speed_index = 1

def draw_speed_box():
    font = pygame.font.Font(None, 24)
    speed_text = f"Time Speed: {current_speed_index + 1}"
    text_surface = font.render(speed_text, True, (255, 255, 255))

    box_width = text_surface.get_width() + 20
    box_height = text_surface.get_height() + 10

    # Position the box at the bottom left corner
    box_rect = pygame.Rect(10, height - box_height - 10, box_width, box_height)
    screen.fill((0, 0, 0), box_rect)
    pygame.draw.rect(screen, (0, 0, 0), box_rect, 1)
    screen.blit(text_surface, (15, height - box_height))


# In[6]:

# Start the plotting thread


# Then enter your main Pygame loop


# MAIN LOOP and CONTROLS

zoom_speed = 0.1
scroll_speed = 10

dragging = False
last_mouse_pos = None
selected_body = None

frame_rate = 60

earth = next(body for body in bodies if body.name == "Freyja")
star_names = [body.name for body in bodies if body.is_star()]

intensity_values = {star_name: [] for star_name in star_names}  # Dictionary to store intensity values


# Start the plotting thread
plotting_thread = threading.Thread(target=real_time_plotting, args=(intensity_values,))
plotting_thread.daemon = True
plotting_thread.start()

# Then enter your main Pygame loop


while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                paused = not paused
            elif event.key == pygame.K_UP: 
                current_speed_index = min(current_speed_index + 1, len(speed_settings) - 1)
            elif event.key == pygame.K_DOWN:
                current_speed_index = max(current_speed_index - 1, 0)
            if event.key == pygame.K_ESCAPE:
                running = False
        elif event.type == pygame.MOUSEBUTTONDOWN:
            if event.button == 1: 
                mouse_pos = np.array(pygame.mouse.get_pos())
                body_clicked = False
                for body in bodies:
                    if body.is_mouse_over(mouse_pos, scale, sun_center):
                        selected_body = body  
                        body_clicked = True
                        break
                if not body_clicked:
                    selected_body = None
            
            if event.button == 1:  
                dragging = True
                last_mouse_pos = np.array(pygame.mouse.get_pos())
            elif event.button == 4:  
                scale *= (1 + zoom_speed)
            elif event.button == 5: 
                scale /= (1 + zoom_speed)
        elif event.type == pygame.MOUSEBUTTONUP:
            if event.button == 1:  
                dragging = False
        elif event.type == pygame.MOUSEMOTION:
            if dragging:
                current_mouse_pos = np.array(pygame.mouse.get_pos())
                mouse_delta = current_mouse_pos - last_mouse_pos
                sun_center += mouse_delta
                last_mouse_pos = current_mouse_pos

    if not running:
        break

    dt = speed_settings[current_speed_index]  

    if not paused:
        for body in bodies:
            body.calculate_acceleration(bodies)
            body.update(dt)

        earth = next(body for body in bodies if body.name == "Freyja")

        for body in bodies:
            if body.is_star():
                intensity = body.calculate_intensity(earth.position, bodies, scale)
                intensity_values[body.name] = intensity_values.get(body.name, []) + [intensity]

            for body in bodies:
                if body.is_star():
                    intensity = body.calculate_intensity(earth.position, bodies, scale)
                    intensity_values[body.name].append(intensity)

            """
        for star_name in star_names:
            intensity = get_intensity_for_star(star_name)
            intensity_values[star_name].append(intensity)

            """

        
    mouse_pos = np.array(pygame.mouse.get_pos())

    # Check if mouse is over any body
    hovered_body = None
    for body in bodies:
        if body.is_mouse_over(mouse_pos, scale, sun_center):
            hovered_body = body
            break
    
    screen.fill((0, 0, 0))
    for body in bodies:
        draw_body(body)

    if selected_body:
        draw_info_box(selected_body)
    elif hovered_body:
        draw_info_box(hovered_body)

    draw_speed_box()

    pygame.display.flip()
    clock.tick(frame_rate)

pygame.quit()



pygame 2.5.2 (SDL 2.28.3, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


Exception in thread Thread-5 (real_time_plotting):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/opt/homebrew/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/3p/14_l9sln149cqbtdjdlt8xww0000gn/T/ipykernel_12351/1209977891.py", line 179, in real_time_plotting
  File "/opt/homebrew/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 2485, in relim
    self._update_line_limits(artist)
  File "/opt/homebrew/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 2332, in _update_line_limits
    path = line.get_path()
           ^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/matplotlib/lines.py", line 1032, in get_path
    self.recache()
  File "/opt/homebrew/lib/py

Error in callback <function _draw_all_if_interactive at 0x11e4aeac0> (for post_execute):


ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (160024,) and arg 1 with shape (160032,).

ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (160024,) and arg 1 with shape (160032,).

<Figure size 640x480 with 1 Axes>